# Act 1 — Cloud Functions: code that runs on events

The simplest compute primitive on GCP. You write a single function, hand it to GCP, and it runs whenever something specific happens — an HTTP request, a Pub/Sub message, a file landing in GCS. No servers to provision, no containers to build (the platform builds one for you), no scaling to configure.

This is the smallest unit of compute on GCP, but it's also the most opinionated. Knowing where its sweet spot is and where it stops being the right answer saves a lot of pain.

## Cloud Functions (2nd gen)

The 2nd-generation Cloud Functions are built on top of Cloud Run plus Eventarc. That's a meaningful architectural detail: a deployed 2nd-gen function *is* a Cloud Run service with a specific entry-point contract and an Eventarc trigger wired up by GCP. Everything Cloud Run can do (longer execution timeouts, concurrent requests per instance, larger machine sizes, traffic splitting) Cloud Functions 2nd gen can do — because it's the same engine underneath.

Gen 1 still exists for backward compatibility. For anything new, use gen 2.

**Two trigger kinds:**

- **HTTP triggers** — the function gets a public or IAM-protected HTTPS URL. The request arrives as a normal Flask/Express-shaped request object.
- **Event triggers (Eventarc)** — the function fires on Pub/Sub messages, GCS object events, Firestore writes, Audit Log entries, Cloud Build status changes, or any Eventarc source.

**Constraints worth knowing:**

- **Execution timeout** — up to 60 minutes (HTTP) or 9 minutes (event-driven), well past the gen-1 9-minute cap.
- **Memory and CPU** — up to 32 GiB and 8 vCPU. Bigger than most teams realise.
- **Concurrency** — gen 2 supports up to 1000 concurrent requests *per instance* (gen 1 was strictly 1-per-instance), so cold starts amortise across requests.

**When Cloud Functions is right:** a small, well-defined reaction to one specific event. "When this file lands, resize it." "When this Pub/Sub message arrives, write to BigQuery." When the function grows past one event source or wants to share libraries with another, you're already on Cloud Run; promote it there explicitly and stop pretending it's still a function.

# Act 2 — Cloud Run: GCP's headline PaaS

Cloud Run is where GCP's centre of gravity lives. If you can package your workload as a container that listens on a port, Cloud Run will run it — scaling to zero when idle, scaling out to thousands of instances under load, billing per millisecond of actual execution, with zero infrastructure to manage.

It sits between Cloud Functions (one event, one function) and GKE (a whole cluster you operate). For most stateless HTTP and event-driven workloads on GCP today, Cloud Run is the right answer; the act of deciding *not* to use it should require a real reason.

## Cloud Run Services — long-running, request-driven

A Cloud Run **Service** is an HTTP-serving workload. You give it a container image; Cloud Run gives you back a stable HTTPS URL.

**Key model decisions Cloud Run hides from you:** how many instances exist, where they run, how requests are routed. You configure:

- **CPU and memory per instance** — up to 8 vCPU, 32 GiB.
- **Concurrency per instance** — default 80. Each instance can handle *N* concurrent requests. This is the big differentiator from Cloud Functions gen 1 and from AWS Lambda — your container amortises cold starts across many requests.
- **Min and max instances** — `min=0` is the default (scale to zero, cold starts on first request); `min=1` keeps one warm at a small cost; `max` caps the blast radius of a traffic spike.
- **CPU allocation** — **request-only** (CPU only while serving) or **always-allocated** (CPU during idle too, needed for background work like async tasks or streaming).

**Cold starts.** The first request after scale-to-zero pays a cold-start cost — pulling the image, starting the container, your app's startup time. Most apps land in 200–800 ms range. If you can't tolerate that on the first request, set `min=1` or use a warmer cron. Cloud Run also supports **startup CPU boost** to shorten cold-start CPU time.

**Traffic splitting.** Every deployment creates a new **revision**. You can split traffic between revisions arbitrarily — 90/10, blue/green, canary. Combined with Cloud Deploy (notebook 13), this is how zero-downtime rollouts work on Cloud Run.

**Networking — VPC connectors and Direct VPC egress.** By default Cloud Run egresses straight to the internet via a Google-managed network. To reach private resources (Cloud SQL with a private IP, internal load balancers, on-prem via VPN), you wire Cloud Run into your VPC. Two ways:

- **Serverless VPC Access connector** — older, requires running a small connector resource (~$10/mo per connector).
- **Direct VPC egress** — newer, no connector resource. The default now for new services.

## Cloud Run Jobs — batch on the same platform

A **Cloud Run Job** runs a container to completion — no port, no HTTP, no "always listening." Use it for: scheduled batch processing, migrations, DB rebuilds, the once-a-day reports, scientific compute tasks.

Jobs share the same container model, IAM, networking, and observability as Services. The differences:

- **No traffic** — the job runs when you trigger it (`gcloud run jobs execute`, Cloud Scheduler cron, Eventarc).
- **Task parallelism** — a single Job execution can fan out to *N* parallel tasks (`--tasks=100`), each with a `CLOUD_RUN_TASK_INDEX` env var for sharding.
- **Timeouts up to 24 hours per task.**

**Sidecars.** Both Services and Jobs support multi-container deployments (`--containers`). Use this for log shipping, proxies (Cloud SQL Auth Proxy is the canonical example), or any sidecar pattern you'd use in Kubernetes.

**Cloud Run for Anthos** is mostly historical — it's been superseded by running Cloud Run on Knative-compatible GKE clusters via the `Service` CRD. Worth knowing it exists; not worth designing around.

# Act 3 — GKE: when you actually need Kubernetes

GKE is GCP's managed Kubernetes. If your workload is genuinely Kubernetes-shaped — multiple cooperating services, stateful workloads, Helm charts you can't easily rewrite, custom operators — GKE is the right answer.

If your workload is *not* Kubernetes-shaped and you're reaching for GKE anyway, the most useful thing this act can do is talk you out of it. Cloud Run handles a startling share of what teams initially think they need a cluster for.

## GKE Standard vs Autopilot — the operational dial

GKE comes in two modes. They're the same Kubernetes API; they differ on how much of the node-and-cluster operation Google does for you.

| Aspect | Standard | Autopilot |
|---|---|---|
| **Node pools** | You define, size, and upgrade them | Google provisions per-pod |
| **Node OS upgrades** | You manage (with auto-upgrade option) | Fully managed |
| **Bin-packing** | Your problem — set pod requests right or waste nodes | Google's problem — pay per pod, not per node |
| **Workload Identity** | Optional (strongly recommended) | Enforced |
| **Pricing** | Pay for nodes (whatever you provision) | Pay per pod CPU/memory/storage used |
| **Sweet spot** | Custom node shapes, GPUs/TPUs, large DaemonSets, low-level tweaks | Standard stateless apps that fit the Autopilot constraints |

**Default to Autopilot for new clusters.** The operational savings are real; the constraints (no privileged containers, limited HostPath, fewer DaemonSet patterns) are the same constraints you should already be enforcing. Standard is the right choice when you genuinely need control over the node shape or DaemonSets that Autopilot won't run.

## GKE networking — VPC-native by default

Two cluster networking modes exist; one is the modern default.

- **VPC-native (alias IP)** — pods and services get IPs from secondary CIDR ranges on the cluster's subnet. Pod IPs are routable inside the VPC. This is the default and the only choice for Autopilot. Required for several modern features (Private Google Access, internal LBs targeting pods directly, NEG-based LB).
- **Routes-based** — legacy. Pods get IPs from a separate range and the cluster manages VPC routes for them. Don't pick this for new clusters.

**Cluster autoscaler** scales node pools up and down based on pending pods. **Node auto-provisioning** (NAP) goes further: it creates *new node pools* with appropriate shapes when no existing pool can fit the pending pod. NAP is the closest Standard mode gets to Autopilot — you give up shape control to gain bin-packing.

## Workload Identity in GKE — the only sensible auth pattern

We met this in notebook 02. Recap because it's the central security primitive on GKE:

- Annotate a Kubernetes Service Account with `iam.gke.io/gcp-service-account: GSA-EMAIL`.
- Grant `roles/iam.workloadIdentityUser` on the Google SA to `serviceAccount:PROJECT.svc.id.goog[NAMESPACE/KSA-NAME]`.
- Pods using the KSA get short-lived Google credentials via the GKE metadata server.

No SA JSON keys. No mounted secrets. No rotation. Autopilot enforces this; on Standard, turn it on at cluster create time. Any other pattern (`GOOGLE_APPLICATION_CREDENTIALS` env var, mounted SA keys) is a security regression in 2026.

## Ingress and the Gateway API

GKE supports both the older `Ingress` resource and the newer `Gateway` API. Both ultimately configure GCP Cloud Load Balancing (notebook 07).

- **GKE Ingress** — single-cluster L7 LB, well-trodden, fine for most apps.
- **Gateway API** — newer, more expressive, supports multi-cluster routing and richer header/path manipulation. The way to go for any new multi-tenant or multi-cluster setup.

The LB itself is the Global External Application LB from notebook 07 — same anycast, same Cloud Armor integration, same Cloud CDN compatibility.

# Act 4 — Plumbing and the choose-what tree

Whichever runtime you pick — Cloud Functions, Cloud Run, or GKE — every container goes through the same two-step pipeline: **build** the image, **store** it somewhere your runtime can pull from. That's Cloud Build and Artifact Registry.

We close the act with a decision tree, because the most common question in a real review is "Cloud Run or GKE?" — and there's a defensible answer.

## Artifact Registry — where images and packages live

**Artifact Registry** replaced the older Container Registry (`gcr.io`). For anything new, use Artifact Registry.

Supported formats:

- **Docker / OCI** — the obvious one. Cloud Run pulls from here; GKE pulls from here.
- **Maven, npm, Python, apt, yum** — language and OS package repos hosted alongside container images.
- **Helm charts** — Kubernetes deployments often store their charts next to the images they reference.

Two features worth knowing:

- **Vulnerability scanning** — Container Analysis scans images on push and continuously, surfacing CVEs in your dependencies. Findings flow into Security Command Center (notebook 11).
- **Remote and virtual repositories** — proxy public registries (Docker Hub, npm) through Artifact Registry to get caching, vulnerability scanning, and policy enforcement on the path.

## Cloud Build — the build pipeline

**Cloud Build** runs container-based steps in sequence (or in parallel). Each step is itself a container; the canonical step images (`gcr.io/cloud-builders/docker`, `gcr.io/cloud-builders/gcloud`, etc.) ship with the tools you need pre-installed.

A `cloudbuild.yaml` is the source of truth:

```yaml
steps:
  - name: gcr.io/cloud-builders/docker
    args: ['build', '-t', '$_IMAGE', '.']
  - name: gcr.io/cloud-builders/docker
    args: ['push', '$_IMAGE']
  - name: gcr.io/google.com/cloudsdktool/cloud-sdk
    args: ['gcloud', 'run', 'deploy', 'app', '--image=$_IMAGE', '--region=us-central1']
substitutions:
  _IMAGE: us-central1-docker.pkg.dev/$PROJECT_ID/app/web:$SHORT_SHA
```

**Triggers** kick off builds on GitHub/GitLab push events, Cloud Source Repos, Pub/Sub messages, or webhook calls. **Private pools** run builds inside your VPC (needed if your build needs access to private resources). Notebook 13 covers Cloud Build + Cloud Deploy as the canonical CD pattern.

## Choose-what — Cloud Functions vs Cloud Run vs GKE

The shortest defensible decision tree:

1. **One specific event, one piece of code?** → Cloud Functions (gen 2).
2. **HTTP service or event-driven worker, no Kubernetes-specific needs?** → Cloud Run.
3. **Multiple cooperating services, stateful workloads, Helm/operators you can't easily rewrite, custom networking?** → GKE (Autopilot first, Standard only if you need node-level control).
4. **Long-running stateful VMs (databases you manage yourself, legacy software that can't containerise)?** → Compute Engine (notebook 03).

**Anti-patterns to recognise:**

- "We need Kubernetes because we're going to scale." — Cloud Run scales to 1000+ instances. Test that ceiling before assuming you need GKE.
- "We need Cloud Run min-instances=10 forever because cold starts." — That's a Compute Engine VM with extra steps. Measure cold starts; consider startup CPU boost first.
- "We're putting our database on GKE." — Almost always wrong. Use Cloud SQL, AlloyDB, Spanner, Firestore, or Memorystore depending on the workload (notebooks 08 and 09).

**Cost shape across runtimes.** Cloud Functions and Cloud Run bill per-request and per-millisecond, so cost tracks usage almost linearly. GKE bills for nodes (Standard) or per-pod resources (Autopilot), so cost tracks *capacity*, not requests. For bursty/low-traffic workloads Cloud Run wins on cost; for steady-state high-traffic workloads with good bin-packing, GKE wins.

## What carries into later chapters

Cloud Run is the default home for new services in the rest of this course. The networking pieces — VPC connectors, internal LBs, Global External Application LB in front — are notebook 06 and 07. The deployment plumbing — Cloud Build, Cloud Deploy, Workload Identity Federation from GitHub Actions — is notebook 13. The observability — Cloud Run request logs, Cloud Trace spans, Cloud Profiler attached — is notebook 12.

The lesson to internalise from this chapter: **GCP's container story is one container model and three places to run it.** Cloud Functions wraps it for events. Cloud Run runs it as a managed service. GKE runs it inside a cluster you (lightly) operate. The container itself — the Dockerfile, the entrypoint, the port — is the same. Choosing between the three is about how much of the surrounding infrastructure you want to own.